In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import pickle
import json

In [3]:

df = pd.read_csv(r'D:\Miini_project2\data\cleaned_data.csv')

In [5]:
# Clean column names
df.columns = [col.strip().replace(' ', '_').replace('(', '').replace(')', '') for col in df.columns]

# Use only features that exist in the dataframe
all_columns = set(df.columns)
categorical_features = [col for col in ['gender', 'category_name', 'payment_method', 'city'] if col in all_columns]
numeric_features = [col for col in ['quantity', 'price', 'age', 'price_per_item'] if col in all_columns]
target = 'price' if 'price' in all_columns else df.columns[-1]  # fallback to last column if needed

print('Categorical features:', categorical_features)
print('Numeric features:', numeric_features)
print('Target:', target)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer([
    ('num', 'passthrough', numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
], remainder='drop')

X = df[categorical_features + numeric_features]
y = df[target]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train_processed, y_train)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2': r2_score(y_test, y_pred)
    }

metrics = evaluate_model(model, X_test_processed, y_test)

artifacts = {
    'model': model,
    'preprocessor': preprocessor,
    'metrics': metrics,
    'feature_names': list(X.columns)
}

Categorical features: ['gender', 'category_name', 'payment_method', 'city']
Numeric features: ['quantity', 'price', 'age', 'price_per_item']
Target: price


In [9]:
with open(r'D:\Miini_project2\data_clean\data_cleaning.ipynb.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

In [11]:

metadata = {
    'model_type': 'RandomForestRegressor',
    'version': '1.0',
    'features': {
        'categorical': categorical_features,
        'numeric': numeric_features
    },
    'performance': metrics
}

with open(r'D:\Miini_project2\appmodel_metaatad.json', 'w') as f:
    json.dump(metadata, f, indent=2)